# PERG-HGP vs HDBSCAN: Google Colab GPU G4 Benchmark

Ce notebook permet d'évaluer les performances en temps de calcul, en précision (ARI) et en exactitude géométrique du clusterer haute-performance **PERG-HGP** sur GPU (T4/L4 de type G4) face à **HDBSCAN**.

## 🛠️ Configuration d'accélération matérielle :
Assurez-vous que l'accélérateur GPU est actif dans Google Colab :
1. Allez dans le menu **Runtime** -> **Change runtime type**.
2. Sélectionnez **T4 GPU** ou **L4 GPU**.
3. Cliquez sur **Save**.

### 1. Installation des dépendances et Clone du Dépôt

In [ ]:
# Installation des dépendances
!pip install hdbscan

import sys
import os

# Clone du dépôt si exécuté sous Google Colab
if not os.path.exists('/content/E-HGP'):
    !git clone https://github.com/Ludwig-H/E-HGP.git /content/E-HGP
    sys.path.append('/content/E-HGP/perg_hgp')
else:
    sys.path.append('/content/E-HGP/perg_hgp')

print("Dépôt cloné et package importable !")

### 2. Vérification du GPU CUDA

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositif de calcul détecté : {device.upper()}")
if device == 'cpu':
    print("❌ WARNING: CUDA GPU non détecté. Veuillez changer le type de runtime de Colab.")
else:
    print(f"✓ GPU NVIDIA actif : {torch.cuda.get_device_name(0)}")

### 3. Imports et Générateurs de Données

In [ ]:
import time
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score
from hdbscan import HDBSCAN
from perg_hgp import PERGHGPClusterer

def make_concentric_spheres(n_samples=10000, noise=0.08):
    n_inner = n_samples // 2
    r_inner = 1.0
    phi = np.random.uniform(0, 2 * np.pi, n_inner)
    theta = np.random.uniform(0, np.pi, n_inner)
    x = r_inner * np.sin(theta) * np.cos(phi) + np.random.normal(0, noise, n_inner)
    y = r_inner * np.sin(theta) * np.sin(phi) + np.random.normal(0, noise, n_inner)
    z = r_inner * np.cos(theta) + np.random.normal(0, noise, n_inner)
    X_inner = np.column_stack([x, y, z])
    y_inner = np.zeros(n_inner)
    
    n_outer = n_samples - n_inner
    r_outer = 4.0
    phi = np.random.uniform(0, 2 * np.pi, n_outer)
    theta = np.random.uniform(0, np.pi, n_outer)
    x = r_outer * np.sin(theta) * np.cos(phi) + np.random.normal(0, noise * 2, n_outer)
    y = r_outer * np.sin(theta) * np.sin(phi) + np.random.normal(0, noise * 2, n_outer)
    z = r_outer * np.cos(theta) + np.random.normal(0, noise * 2, n_outer)
    X_outer = np.column_stack([x, y, z])
    y_outer = np.ones(n_outer)
    
    return np.vstack([X_inner, X_outer]).astype(np.float32), np.concatenate([y_inner, y_outer])

print("Fonctions de génération prêtes !")

### 4. Benchmarks Croissants en Taille et en Difficulté

Nous testons sur des tailles de $10\,000$ à $100\,000$ points, et faisons varier l'ordre $K$, le coefficient géométrique `expZ` et `K_rho` pour évaluer leur impact sur l'ARI et le taux de fallback de PERG-HGP comparé à HDBSCAN.

In [ ]:
scenarios = [
    ("3D Blobs (10k)", lambda: make_blobs(n_samples=10000, centers=5, n_features=3, cluster_std=1.0, random_state=42), 100),
    ("Concentric Spheres (10k)", lambda: make_concentric_spheres(n_samples=10000, noise=0.08), 100),
    ("3D Blobs (50k)", lambda: make_blobs(n_samples=50000, centers=8, n_features=3, cluster_std=1.2, random_state=42), 500),
    ("Concentric Spheres (50k)", lambda: make_concentric_spheres(n_samples=50000, noise=0.08), 500),
    ("3D Blobs (100k)", lambda: make_blobs(n_samples=100000, centers=12, n_features=3, cluster_std=1.5, random_state=42), 1000),
]

hgp_param_sets = [
    {"K": 2, "expZ": 2.0, "K_rho": 32},
    {"K": 5, "expZ": 3.0, "K_rho": 48},
    {"K": 10, "expZ": 1.5, "K_rho": 64},
]

print("| Dataset | Method | K | expZ | K_rho | Time (s) | ARI | Clusters Found | Clustered % | Fallback % |")
print("| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |")

for name, data_fn, min_size in scenarios:
    X, y = data_fn()
    X = X.astype(np.float32)
    
    # HDBSCAN
    t0 = time.time()
    hdb = HDBSCAN(min_cluster_size=min_size)
    labels_hdb = hdb.fit_predict(X)
    t_hdb = time.time() - t0
    ari_hdb = adjusted_rand_score(y, labels_hdb)
    pct_hdb = np.sum(labels_hdb >= 0) / len(labels_hdb) * 100
    n_cl_hdb = len(np.unique(labels_hdb[labels_hdb >= 0]))
    print(f"| {name} | HDBSCAN | - | - | - | {t_hdb:.4f}s | {ari_hdb:.4f} | {n_cl_hdb} | {pct_hdb:.2f}% | - |")
    
    # PERG-HGP
    for params in hgp_param_sets:
        t0 = time.time()
        perg = PERGHGPClusterer(
            K=params["K"],
            expZ=params["expZ"],
            K_rho=params["K_rho"],
            min_cluster_size=min_size,
            device=device
)
        labels_perg = perg.fit_predict(X)
        t_perg = time.time() - t0
        ari_perg = adjusted_rand_score(y, labels_perg)
        pct_perg = np.sum(labels_perg >= 0) / len(labels_perg) * 100
        n_cl_perg = len(np.unique(labels_perg[labels_perg >= 0]))
        fb_rate = perg.exactness_report_.get('global_fallback_rate', 0.0) * 100
        print(f"| {name} | PERG-HGP | {params['K']} | {params['expZ']} | {params['K_rho']} | {t_perg:.4f}s | {ari_perg:.4f} | {n_cl_perg} | {pct_perg:.2f}% | {fb_rate:.1f}% |")

### 5. Benchmark à 1 Million de Points

Nous comparons ici la scalabilité sur 1 million de points.

In [ ]:
N_large = 1000000
print(f"Génération de 1 Million de points 3D...")
X_large, y_large = make_blobs(n_samples=N_large, centers=20, n_features=3, cluster_std=1.0, random_state=42)
X_large = X_large.astype(np.float32)

print("\n--- Running HDBSCAN (1M points) ---")
t0 = time.time()
hdb = HDBSCAN(min_cluster_size=1000, core_dist_n_jobs=-1)
labels_hdb = hdb.fit_predict(X_large)
t_hdb = time.time() - t0
sub_idx = np.random.choice(N_large, size=50000, replace=False)
ari_hdb = adjusted_rand_score(y_large[sub_idx], labels_hdb[sub_idx])
pct_hdb = np.sum(labels_hdb >= 0) / len(labels_hdb) * 100
n_cl_hdb = len(np.unique(labels_hdb[labels_hdb >= 0]))
print(f"HDBSCAN terminé en {t_hdb:.2f}s | ARI: {ari_hdb:.4f} | Clusters: {n_cl_hdb} | % Classé: {pct_hdb:.2f}%")

print("\n--- Running PERG-HGP (1M points) ---")
t0 = time.time()
perg = PERGHGPClusterer(K=10, min_cluster_size=1000, device=device)
labels_perg = perg.fit_predict(X_large)
t_perg = time.time() - t0
ari_perg = adjusted_rand_score(y_large[sub_idx], labels_perg[sub_idx])
pct_perg = np.sum(labels_perg >= 0) / len(labels_perg) * 100
n_cl_perg = len(np.unique(labels_perg[labels_perg >= 0]))
print(f"PERG-HGP terminé en {t_perg:.2f}s | ARI: {ari_perg:.4f} | Clusters: {n_cl_perg} | % Classé: {pct_perg:.2f}%")
print("Rapport d'Exactitude de PERG-HGP:")
for k, v in perg.exactness_report_.items():
    print(f"  - {k}: {v}")

### 6. Le fameux Test de Scalabilité Cible : 30 000 000 de Points ($K=10$)

Cette dernière cellule exécute le test de stress maximal sur 30 millions de points. Grâce à notre streaming KNN-Gibbs par blocs et le chunking interne de la recherche de grille, la consommation de VRAM GPU et RAM reste parfaitement contrôlée.

In [ ]:
N_target = 30000000
print(f"Génération du jeu de données cible de {N_target} points...")
X_target, y_target = make_blobs(n_samples=N_target, centers=50, n_features=3, cluster_std=1.0, random_state=42)
X_target = X_target.astype(np.float32)

print(f"\nAllocation de PERGHGPClusterer (K=10, device={device})...")
# Note : Les budgets par défaut sont optimisés pour les instances avec beaucoup de RAM (160 Go).
# Si vous manquez de mémoire système, vous pouvez restreindre max_unique_facets pour utiliser le mode Out-of-Core disque.
clusterer = PERGHGPClusterer(
    K=10,
    expZ=2.0,
    min_cluster_size=10000,
    device=device
)

print("Exécution de fit_predict...")
t0 = time.time()
labels = clusterer.fit_predict(X_target)
t_total = time.time() - t0

print(f"\n🎉 PERG-HGP 30M points exécuté avec succès en {t_total:.2f} secondes ({t_total/60:.2f} minutes) !")
n_cl = len(np.unique(labels[labels >= 0]))
pct_cl = np.sum(labels >= 0) / len(labels) * 100
print(f"Clusters identifiés : {n_cl} | Points classés % : {pct_cl:.2f}%")

print("\nRapport d'Exactitude de PERG-HGP (30M points) :")
for k, v in clusterer.exactness_report_.items():
    print(f"  - {k}: {v}")